# 원본 병변 보존 crop — 캐글 무료 파일럿

학습 20,000장 / 검증 4,000장. 일반 random crop과 병변 보존 crop을 **동일 초기 가중치**로 비교합니다. **1단계 정상/이상 분류**이며 holdout은 사용하지 않습니다.

1. `dogskin_safe_crop_pilot.zip`을 **Private Dataset**으로 업로드한 뒤 Add Input.
2. 이 노트북을 Import. **GPU T4 x2**, **Internet On** 설정(실제 학습은 한 장 사용). 첫 실행에서 사전학습 가중치를 받습니다.
3. 남은 GPU 9시간이면 아래 `HOURS=7.0` 유지. **Save Version → Save & Run All**로 한 번 실행합니다.
4. Output의 `safe_crop_pilot_resume.zip`을 보관합니다. 다음 세션에서 이 ZIP을 Add Input하면 이어집니다.

T4 x2 선택 시에도 한 장만 사용합니다. 업로드·입력 준비 동안 GPU는 끄고, 편집 세션과 Save & Run All에서 전체 학습을 두 번 돌리지 마세요.

[캐글 실행 안내](https://www.kaggle.com/docs/notebooks) · [GPU 사용 안내](https://www.kaggle.com/docs/efficient-gpu-usage)

**CUDA 호환성 수정판:** `no kernel image is available` 오류를 겪었다면 기존 Dataset은 그대로 두고 이 노트북만 다시 Import하세요. 일부 최신 PyTorch CUDA 빌드는 P100을 지원하지 않습니다. 첫 셀에서 실제 ResNet50 FP32/AMP 순전파·역전파를 검사합니다. P100 대신 T4를 우선 사용합니다.

첫 학습 전에 실패해 완료 epoch가 없는 이전 실행의 재개 ZIP은 연결하지 않고 새로 시작하세요. 완료된 epoch가 있다면 기존 파일을 보존하고 환경/체크포인트 호환성을 먼저 확인해야 합니다.

[PyTorch GPU 지원 변경 안내](https://dev-discuss.pytorch.org/t/cuda-toolkit-version-and-architecture-support-update-maxwell-and-pascal-architecture-support-removed-in-cuda-12-8-and-12-9-builds/3128)


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
import zipfile

HOURS = 7.0
EPOCHS = 5
BATCH_SIZE = 32
WORKERS = 2
# 재개 파일이 여러 개일 때 원하는 ZIP 또는 protocol.json이 있는 폴더를 지정합니다.
RESUME_FROM = None
INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/safe_crop_pilot')
SCRATCH = Path('/kaggle/temp/safe_crop_pilot_data')

import torch
assert torch.cuda.is_available(), 'Notebook Settings에서 GPU를 켜주세요.'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('이번 실행 예산:', HOURS, '시간 / 두 방법 각각 최대', EPOCHS, 'epoch')

# GPU가 표시되어도 설치된 CUDA wheel이 해당 GPU를 지원하지 않을 수 있습니다.
# 새 프로세스에서 실제 모델 연산을 확인해 노트북 커널에 CUDA 오류 상태가 남지 않게 합니다.
GPU_PROBE = r"""
import torch
import torchvision
from torchvision.models import resnet50
print('GPU:', torch.cuda.get_device_name(0), flush=True)
print('Compute capability:', torch.cuda.get_device_capability(0), flush=True)
print('PyTorch / torchvision:', torch.__version__, torchvision.__version__, flush=True)
print('CUDA build:', torch.version.cuda, flush=True)
print('Compiled architectures:', torch.cuda.get_arch_list(), flush=True)
torch.manual_seed(42)
model = resnet50(weights=None).cuda().train()
x = torch.randn(2, 3, 64, 64, device='cuda')
y = torch.tensor([0, 1], device='cuda')
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)
for amp in (False, True):
    optimizer.zero_grad(set_to_none=True)
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    with torch.autocast(device_type='cuda', enabled=amp):
        loss = torch.nn.functional.cross_entropy(model(x), y)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    torch.cuda.synchronize()
    assert torch.isfinite(loss).item(), 'Nonfinite GPU probe loss'
    print('ResNet50 forward/backward/optimizer OK; AMP =', amp, flush=True)
"""
probe_result = subprocess.run([sys.executable, '-u', '-c', GPU_PROBE],
                              capture_output=True, text=True, timeout=120)
print(probe_result.stdout)
if probe_result.returncode:
    print(probe_result.stderr)
    raise RuntimeError(
        'GPU 연산 사전 검사 실패. 학습은 시작하지 않았습니다. '
        'P100이면 Accelerator를 GPU T4 x2로 바꾸고 새 세션에서 다시 실행하세요. '
        'T4에서도 실패하면 위 GPU 이름·CUDA build·Compiled architectures 출력을 확인해주세요.'
    )


## 입력 찾기

Dataset이 이미 풀려 있으면 `/kaggle/input`에서 바로 읽습니다. ZIP만 붙어 있으면 임시 공간에 풉니다. 사진을 Output 폴더에 복사하지 않습니다. 패키지에 실행 코드가 들어 있어 git clone이나 API 키가 필요 없습니다.


In [ ]:
def safe_extract(archive, destination):
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():
            path = Path(name)
            if path.is_absolute() or '..' in path.parts:
                raise ValueError('ZIP 안에 잘못된 경로가 있습니다.')
        destination.mkdir(parents=True, exist_ok=True)
        z.extractall(destination)

manifests = sorted(INPUT.rglob('pilot_manifest.parquet'))
if not manifests:
    archives = sorted(INPUT.rglob('dogskin_safe_crop_pilot.zip'))
    assert len(archives) == 1, '파일럿 Dataset 하나를 Add Input해주세요.'
    safe_extract(archives[0], SCRATCH)
    manifests = sorted(SCRATCH.rglob('pilot_manifest.parquet'))
assert len(manifests) == 1, '파일럿 Dataset이 여러 개입니다. 하나만 연결해주세요.'
DATA = manifests[0].parent
metadata = json.loads((DATA / 'pilot_package.json').read_text())
print('데이터:', DATA)
print('학습:', metadata['train_rows'], '검증:', metadata['val_rows'], 'holdout:', metadata['holdout_rows'])
assert metadata['holdout_rows'] == 0
print('원본 JPG:', round(metadata['jpeg_bytes'] / 1024**3, 2), 'GiB')


## 이전 실행 복원

첫 실행은 재개 파일 없이 진행합니다. 재개할 때 이전 Output의 `safe_crop_pilot_resume.zip`을 Add Input하세요. ZIP이 자동으로 풀린 경우도 지원합니다. 현재 작업 폴더에 체크포인트가 있으면 그 상태를 계속 사용합니다.


In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
if not (OUTPUT / 'protocol.json').exists():
    candidates = [Path(RESUME_FROM)] if RESUME_FROM else sorted(INPUT.rglob('safe_crop_pilot_resume.zip'))
    if not candidates:
        candidates = [path.parent for path in sorted(INPUT.rglob('protocol.json'))
                      if (path.parent / 'initial.pt').exists()]
    assert len(candidates) <= 1, '재개 파일이 여러 개입니다. RESUME_FROM을 지정해주세요.'
    if candidates:
        previous = candidates[0]
        if previous.is_dir():
            shutil.copytree(previous, OUTPUT, dirs_exist_ok=True)
        else:
            safe_extract(previous, OUTPUT)
        assert (OUTPUT / 'protocol.json').exists(), '재개 ZIP 구조를 확인해주세요.'
        print('이전 실행 복원:', previous)
    else:
        print('새 실행: 두 조건이 공유할 초기 가중치를 준비합니다.')
else:
    print('현재 작업 폴더에서 이어서 실행합니다.')


## 두 방식 비교 실행

한 epoch씩 번갈아 실행합니다. 첫 epoch가 끝나면 실제 소요 시간과 예상 총시간이 출력됩니다. 예산에 다음 epoch 쌍이 들어가지 않으면 체크포인트와 결과를 남기고 종료합니다. 도중 중단된 epoch는 다음 실행에서 다시 시작합니다.

`HOURS`만 다음 세션의 남은 시간에 맞게 바꿀 수 있습니다. 재개 시 다른 설정과 Dataset 버전은 유지하세요. 플랫폼의 강제 종료까지 Output 보존을 보장하지는 않으므로, 여유 시간과 정상 종료를 확보합니다.


In [ ]:
import os

# 원본 JPG 일부가 끝이 잘려 있어 PIL이 OSError(image file is truncated)로 죽습니다.
# 하위 프로세스와 DataLoader 워커에도 적용되도록 sitecustomize 로 넣습니다.
PIL_PATCH = Path('/kaggle/working/pil_patch')
PIL_PATCH.mkdir(parents=True, exist_ok=True)
(PIL_PATCH / 'sitecustomize.py').write_text(
    'from PIL import ImageFile\nImageFile.LOAD_TRUNCATED_IMAGES = True\n')
env = dict(os.environ)
env['PYTHONPATH'] = os.pathsep.join([str(PIL_PATCH), env.get('PYTHONPATH', '')]).rstrip(os.pathsep)

command = [sys.executable, '-u', str(DATA / 'tools/kaggle_safe_crop.py'),
           '--data', str(DATA), '--out', str(OUTPUT),
           '--hours', str(HOURS), '--epochs', str(EPOCHS),
           '--batch-size', str(BATCH_SIZE), '--workers', str(WORKERS)]
subprocess.run(command, check=True, cwd=DATA, env=env)


## 결과와 재개 파일

`comparison.json`은 두 방식 모두 끝낸 **마지막 공통 epoch**를 비교합니다. macro F1과 이상 recall뿐 아니라 정상 specificity도 함께 보세요. 단일 seed 예비 실험으로 작은 차이를 확정 개선으로 해석하지 않습니다. `*_best.pt`의 최고 epoch가 서로 달라도, 아래 비교는 공통 epoch를 기준으로 합니다.


In [ ]:
import pandas as pd
comparison = json.loads((OUTPUT / 'comparison.json').read_text())
print('완료 epoch:', comparison['completed_epochs'])
print('비교 epoch:', comparison['common_epoch'])
if comparison['comparison']:
    display(pd.DataFrame(comparison['comparison']).T)
else:
    print('아직 공통 완료 epoch가 없습니다. 체크포인트를 이어서 실행해주세요.')
display(pd.read_csv(OUTPUT / 'history.csv'))
print('재개 ZIP:', OUTPUT.parent / 'safe_crop_pilot_resume.zip')
print('Output의 ZIP과 comparison.json을 다운로드해 보관하세요.')
